In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [10]:
import pandas as pd

DATA_PATH = "../data/processed/technical_feature_matrix.csv"

df = pd.read_csv(DATA_PATH)

df.shape

(9776, 23)

In [11]:
X = df.drop(columns=["label"])
y = df["label"]

print("X:", X.shape)
print("y:", y.shape)

X: (9776, 22)
y: (9776,)


In [12]:
df

,semantic_similarity,tfidf_similarity,word_overlap,resume_skill_count,job_skill_count,matched_skill_count,skill_match_ratio,skill_coverage,resume_education_level,job_education_level,...,experience_match,resume_core_cs_count,job_core_cs_count,resume_degree_count,job_degree_count,resume_length,job_description_length,resume_word_count,job_word_count,label
0,0.507104,0.056482,0.133333,4,0,0,0.000000,0.000000,5,0,...,1.0,1,0,1,0,286,696,37,99,1
1,0.562818,0.299446,0.750000,14,4,4,1.000000,0.285714,3,0,...,1.0,4,2,1,0,2506,107,333,16,1
2,0.523189,0.054792,0.133333,5,0,0,0.000000,0.000000,2,0,...,1.0,1,0,0,0,287,696,37,99,1
3,0.529498,0.234125,0.533333,7,0,0,0.000000,0.000000,4,0,...,1.0,1,0,2,0,3143,91,401,16,1
4,0.471209,0.047656,0.117647,1,0,0,0.000000,0.000000,0,3,...,1.0,0,0,0,1,277,616,37,90,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9771,0.537629,0.358228,0.476190,9,0,0,0.000000,0.000000,3,0,...,1.0,1,0,1,0,2655,157,343,24,0
9772,0.711359,0.186097,0.533333,3,0,0,0.000000,0.000000,3,0,...,1.0,0,0,1,0,3242,105,428,15,1
9773,0.585595,0.203442,0.315789,2,0,0,0.000000,0.000000,0,0,...,1.0,0,0,0,0,466,125,52,18,1
9774,0.588449,0.085541,0.223881,1,3,1,0.333333,1.000000,5,0,...,1.0,0,1,1,0,306,664,38,90,0


### Train/Test Splitting

In [13]:
NORMALIZED_PATH = "../data/processed/technical_normalized_dataset.csv"

normalized_df = pd.read_csv(NORMALIZED_PATH)

print("Normalized dataset shape:", normalized_df.shape)

Normalized dataset shape: (9776, 11)


In [14]:
assert len(normalized_df) == len(df)

print("✓ Row counts match")

✓ Row counts match


In [15]:
assert normalized_df["label"].equals(df["label"])

print("✓ Labels match")

✓ Labels match


In [16]:
groups = normalized_df["job_description"].fillna("").astype(str)

print("Total rows:", len(groups))
print("Unique job descriptions:", groups.nunique())

Total rows: 9776
Unique job descriptions: 2525


In [19]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 7875
Testing samples : 1901


For no JD leakage

In [20]:
train_jobs = set(
    groups.iloc[train_idx]
)

test_jobs = set(
    groups.iloc[test_idx]
)

overlap = train_jobs.intersection(test_jobs)

print("Train job descriptions:", len(train_jobs))
print("Test job descriptions :", len(test_jobs))
print("Overlapping job descriptions:", len(overlap))

assert len(overlap) == 0

print("✓ No job-description leakage")

Train job descriptions: 2020
Test job descriptions : 505
Overlapping job descriptions: 0
✓ No job-description leakage


In [21]:
print("Training label distribution:")
print(y_train.value_counts())
print()

print("Training label percentages:")
print(y_train.value_counts(normalize=True).mul(100).round(2))
print()

print("Test label distribution:")
print(y_test.value_counts())
print()

print("Test label percentages:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

Training label distribution:
label
0    4095
1    3780
Name: count, dtype: int64

Training label percentages:
label
0    52.0
1    48.0
Name: proportion, dtype: float64

Test label distribution:
label
1    978
0    923
Name: count, dtype: int64

Test label percentages:
label
1    51.45
0    48.55
Name: proportion, dtype: float64


### Logistic Regression

In [22]:
logistic_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

print("Training Logistic Regression...")

logistic_model.fit(
    X_train,
    y_train
)

print("✓ Training complete")

Training Logistic Regression...
✓ Training complete


In [24]:
#Predictions

y_pred = logistic_model.predict(X_test)

y_prob = logistic_model.predict_proba(X_test)[:, 1]

In [25]:
#Evaluate

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

print("=" * 60)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

LOGISTIC REGRESSION RESULTS
Accuracy : 0.5034
Precision: 0.5209
Recall   : 0.4335
F1 Score : 0.4732
ROC-AUC  : 0.5086


In [26]:
#Classification report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Rejected",
            "Accepted"
        ]
    )
)

cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:")
print(cm)

              precision    recall  f1-score   support

    Rejected       0.49      0.58      0.53       923
    Accepted       0.52      0.43      0.47       978

    accuracy                           0.50      1901
   macro avg       0.51      0.51      0.50      1901
weighted avg       0.51      0.50      0.50      1901

Confusion Matrix:
[[533 390]
 [554 424]]


Findings : Yeah this model is basically random and theres no point in this

Our current 22-feature representation isn't giving Logistic Regression useful linear separation between accepted/rejected candidates.

#### Feature Groups

In [27]:
SEMANTIC_FEATURES = [
    "semantic_similarity"
]

LEXICAL_FEATURES = [
    "tfidf_similarity",
    "word_overlap"
]

EXPLICIT_FEATURES = [
    "resume_skill_count",
    "job_skill_count",
    "matched_skill_count",
    "skill_match_ratio",
    "skill_coverage",
    "resume_education_level",
    "job_education_level",
    "education_match",
    "resume_experience_years",
    "job_required_experience_years",
    "experience_match",
    "resume_core_cs_count",
    "job_core_cs_count",
    "resume_degree_count",
    "job_degree_count",
    "resume_length",
    "job_description_length",
    "resume_word_count",
    "job_word_count"
]

In [28]:
print("Semantic:", len(SEMANTIC_FEATURES))
print("Lexical :", len(LEXICAL_FEATURES))
print("Explicit:", len(EXPLICIT_FEATURES))

assert len(
    SEMANTIC_FEATURES
    + LEXICAL_FEATURES
    + EXPLICIT_FEATURES
) == 22

print("✓ Feature groups contain all 22 features")

Semantic: 1
Lexical : 2
Explicit: 19
✓ Feature groups contain all 22 features


In [29]:
def evaluate_logistic_regression(
    feature_columns,
    experiment_name
):
    X_train_exp = X_train[feature_columns]
    X_test_exp = X_test[feature_columns]

    model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])

    model.fit(
        X_train_exp,
        y_train
    )

    predictions = model.predict(
        X_test_exp
    )

    probabilities = model.predict_proba(
        X_test_exp
    )[:, 1]

    results = {
        "experiment": experiment_name,
        "features": len(feature_columns),
        "accuracy": accuracy_score(
            y_test,
            predictions
        ),
        "precision": precision_score(
            y_test,
            predictions
        ),
        "recall": recall_score(
            y_test,
            predictions
        ),
        "f1": f1_score(
            y_test,
            predictions
        ),
        "roc_auc": roc_auc_score(
            y_test,
            probabilities
        )
    }

    return results

In [30]:
experiments = [

    (
        "Semantic",
        SEMANTIC_FEATURES
    ),

    (
        "Lexical",
        LEXICAL_FEATURES
    ),

    (
        "Explicit",
        EXPLICIT_FEATURES
    ),

    (
        "Semantic + Lexical",
        SEMANTIC_FEATURES + LEXICAL_FEATURES
    ),

    (
        "Semantic + Explicit",
        SEMANTIC_FEATURES + EXPLICIT_FEATURES
    ),

    (
        "Lexical + Explicit",
        LEXICAL_FEATURES + EXPLICIT_FEATURES
    ),

    (
        "All Features",
        SEMANTIC_FEATURES
        + LEXICAL_FEATURES
        + EXPLICIT_FEATURES
    )
]


results = []

for name, features in experiments:

    print(
        f"Running: {name}"
    )

    result = evaluate_logistic_regression(
        features,
        name
    )

    results.append(result)

Running: Semantic
Running: Lexical
Running: Explicit
Running: Semantic + Lexical
Running: Semantic + Explicit
Running: Lexical + Explicit
Running: All Features


c:\Users\BHAVANI\anaconda3\envs\ai-hire\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [31]:
results_df = pd.DataFrame(results)

results_df

,experiment,features,accuracy,precision,recall,f1,roc_auc
0,Semantic,1,0.485534,0.000000,0.000000,0.000000,0.527197
1,Lexical,2,0.493951,0.518692,0.226994,0.315789,0.496522
2,Explicit,19,0.499737,0.518934,0.378323,0.437611,0.499327
3,Semantic + Lexical,3,0.501315,0.541436,0.200409,0.292537,0.521373
4,Semantic + Explicit,20,0.501315,0.521490,0.372188,0.434368,0.506242
5,Lexical + Explicit,21,0.500789,0.517835,0.430470,0.470128,0.502631
6,All Features,22,0.503419,0.520885,0.433538,0.473214,0.508644


In [32]:
results_df.sort_values(
    by="roc_auc",
    ascending=False
).reset_index(drop=True)

,experiment,features,accuracy,precision,recall,f1,roc_auc
0,Semantic,1,0.485534,0.000000,0.000000,0.000000,0.527197
1,Semantic + Lexical,3,0.501315,0.541436,0.200409,0.292537,0.521373
2,All Features,22,0.503419,0.520885,0.433538,0.473214,0.508644
3,Semantic + Explicit,20,0.501315,0.521490,0.372188,0.434368,0.506242
4,Lexical + Explicit,21,0.500789,0.517835,0.430470,0.470128,0.502631
5,Explicit,19,0.499737,0.518934,0.378323,0.437611,0.499327
6,Lexical,2,0.493951,0.518692,0.226994,0.315789,0.496522


Yeahh no this model is not the best for this

Or we have to clean our dataset a bit

### Random Forest

In [33]:
from sklearn.ensemble import RandomForestClassifier

In [34]:
def evaluate_random_forest(
    feature_columns,
    experiment_name
):
    X_train_exp = X_train[feature_columns]
    X_test_exp = X_test[feature_columns]

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    model.fit(
        X_train_exp,
        y_train
    )

    predictions = model.predict(
        X_test_exp
    )

    probabilities = model.predict_proba(
        X_test_exp
    )[:, 1]

    return {
        "experiment": experiment_name,
        "features": len(feature_columns),
        "accuracy": accuracy_score(
            y_test,
            predictions
        ),
        "precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_test,
            probabilities
        )
    }

In [35]:
rf_results = []

for name, features in experiments:

    print(
        f"Running Random Forest: {name}"
    )

    result = evaluate_random_forest(
        features,
        name
    )

    rf_results.append(result)

rf_results_df = pd.DataFrame(
    rf_results
)

rf_results_df.sort_values(
    by="roc_auc",
    ascending=False
).reset_index(drop=True)

Running Random Forest: Semantic
Running Random Forest: Lexical
Running Random Forest: Explicit
Running Random Forest: Semantic + Lexical
Running Random Forest: Semantic + Explicit
Running Random Forest: Lexical + Explicit
Running Random Forest: All Features


,experiment,features,accuracy,precision,recall,f1,roc_auc
0,Semantic + Explicit,20,0.571278,0.596679,0.514315,0.552444,0.615031
1,All Features,22,0.560231,0.582944,0.510225,0.544166,0.604328
2,Explicit,19,0.559179,0.588384,0.476483,0.526554,0.601868
3,Lexical + Explicit,21,0.562862,0.584971,0.517382,0.549105,0.597796
4,Semantic + Lexical,3,0.525513,0.538540,0.542945,0.540733,0.559710
5,Lexical,2,0.522883,0.537018,0.526585,0.531750,0.533168
6,Semantic,1,0.510258,0.525627,0.492843,0.508707,0.518223


### XGBoost

In [38]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import pandas as pd



In [45]:
def run_xgboost_experiment(
    feature_names,
    feature_set_name
):

    X_train_subset = X_train[
        feature_names
    ]

    X_test_subset = X_test[
        feature_names
    ]

    # --------------------------------------------------------
    # Create model
    # --------------------------------------------------------

    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss"
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model.fit(
        X_train_subset,
        y_train
    )

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    y_pred = model.predict(
        X_test_subset
    )

    y_prob = model.predict_proba(
        X_test_subset
    )[:, 1]

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    # --------------------------------------------------------
    # Return results
    # --------------------------------------------------------

    return {
        "features": feature_set_name,
        "feature_count": len(feature_names),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "model": model
    }

In [46]:
xgb_results = []

for name, features in experiments:
    print(
            f"Running Random Forest: {name}"
        )
    result = run_xgboost_experiment(
        features,
        name
    )

    xgb_results.append(
        result
    )

Running Random Forest: Semantic
Running Random Forest: Lexical
Running Random Forest: Explicit
Running Random Forest: Semantic + Lexical
Running Random Forest: Semantic + Explicit
Running Random Forest: Lexical + Explicit
Running Random Forest: All Features


In [52]:

xgb_results_df = pd.DataFrame(
    xgb_results
)

xgb_results_df.sort_values(
    by="roc_auc",
    ascending=False
).reset_index(drop=True)

,features,feature_count,accuracy,precision,recall,f1,roc_auc,model
0,Semantic + Explicit,20,0.569174,0.603922,0.472393,0.530120,0.624832,"XGBClassifier(base_score=None, booster=None, c..."
1,All Features,22,0.568648,0.590805,0.525562,0.556277,0.616808,"XGBClassifier(base_score=None, booster=None, c..."
2,Explicit,19,0.571804,0.596698,0.517382,0.554217,0.616334,"XGBClassifier(base_score=None, booster=None, c..."
3,Lexical + Explicit,21,0.561284,0.588670,0.488753,0.534078,0.607494,"XGBClassifier(base_score=None, booster=None, c..."
4,Semantic + Lexical,3,0.529195,0.560408,0.393661,0.462462,0.565118,"XGBClassifier(base_score=None, booster=None, c..."
5,Lexical,2,0.533403,0.559790,0.435583,0.489937,0.550103,"XGBClassifier(base_score=None, booster=None, c..."
6,Semantic,1,0.512362,0.548757,0.293456,0.382412,0.535348,"XGBClassifier(base_score=None, booster=None, c..."


### Gradient Boost

In [48]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
import pandas as pd


In [49]:
def run_gradient_boosting(
    X_train,
    X_test,
    y_train,
    y_test,
    feature_columns,
    feature_set_name
):
    """
    Train and evaluate a Gradient Boosting classifier
    using a specific feature set.
    """

    # --------------------------------------------------------
    # Select features
    # --------------------------------------------------------

    X_train_selected = X_train[
        feature_columns
    ]

    X_test_selected = X_test[
        feature_columns
    ]

    # --------------------------------------------------------
    # Create model
    # --------------------------------------------------------

    model = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model.fit(
        X_train_selected,
        y_train
    )

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    y_pred = model.predict(
        X_test_selected
    )

    y_prob = model.predict_proba(
        X_test_selected
    )[:, 1]

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    # --------------------------------------------------------
    # Return results
    # --------------------------------------------------------

    return {
        "features": feature_set_name,
        "feature_count": len(feature_columns),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "model": model,
    }

In [54]:
gb_results = []

for name, features in experiments:
    print(
            f"Running Gradient Boost: {name}"
        )
    result = run_gradient_boosting(
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            feature_columns=features,
            feature_set_name=name
        )

    gb_results.append(result)

Running Gradient Boost: Semantic
Running Gradient Boost: Lexical
Running Gradient Boost: Explicit
Running Gradient Boost: Semantic + Lexical
Running Gradient Boost: Semantic + Explicit
Running Gradient Boost: Lexical + Explicit
Running Gradient Boost: All Features


In [55]:
gb_results_df = pd.DataFrame(
    gb_results
)

gb_results_df.sort_values(
    by="roc_auc",
    ascending=False
).reset_index(drop=True)

,features,feature_count,accuracy,precision,recall,f1,roc_auc,model
0,Semantic + Explicit,20,0.568648,0.599747,0.485685,0.536723,0.629280,([DecisionTreeRegressor(criterion='friedman_ms...
1,All Features,22,0.574435,0.600476,0.516360,0.555250,0.625934,([DecisionTreeRegressor(criterion='friedman_ms...
2,Explicit,19,0.558653,0.585276,0.487730,0.532069,0.612361,([DecisionTreeRegressor(criterion='friedman_ms...
3,Lexical + Explicit,21,0.564966,0.584736,0.532720,0.557517,0.610117,([DecisionTreeRegressor(criterion='friedman_ms...
4,Semantic + Lexical,3,0.540242,0.579511,0.387526,0.464461,0.571588,([DecisionTreeRegressor(criterion='friedman_ms...
5,Lexical,2,0.534982,0.558750,0.457055,0.502812,0.550452,([DecisionTreeRegressor(criterion='friedman_ms...
6,Semantic,1,0.528669,0.579767,0.304703,0.399464,0.544973,([DecisionTreeRegressor(criterion='friedman_ms...
